<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/direct_path_patching_ioi.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Direct Path Patching Demo

This notebook demonstrates **direct path patching** — a technique for isolating the direct information flow between specific attention heads in a transformer.

## Background

Standard activation patching replaces the full residual stream at a layer, which affects *all* downstream components simultaneously. This tells you that *some* component at a given layer matters, but cannot isolate which specific head-to-head edge carries the signal.

**Direct path patching** isolates a single causal edge: it patches only the contribution of source head A into the query/key/value input of destination head B, leaving every other component's view of A's output unchanged.

We validate on the **Indirect Object Identification (IOI)** task from Wang et al. 2022:
- Clean: *"When Mary and John went to the store, John gave a drink to"* → **Mary**
- Corrupted: *"When Mary and John went to the store, Mary gave a drink to"* → **John**

Metric: normalised logit diff (0 = corrupted baseline, 1 = clean baseline).

## Setup

In [ ]:
# NBVAL_IGNORE_OUTPUT
import os
try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    %pip install transformer_lens
except:
    IN_COLAB = False
    print("Running as a Jupyter notebook")

In [ ]:
import torch
from transformer_lens import HookedTransformer
from transformer_lens.direct_path_patching import get_act_patch_direct_path

## Load Model

In [ ]:
model = HookedTransformer.from_pretrained(
    "gpt2",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
)
model.eval()
print(f"Loaded GPT-2 small: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## Define IOI Task

In [ ]:
CLEAN_PROMPT = "When Mary and John went to the store, John gave a drink to"
CORRUPTED_PROMPT = "When Mary and John went to the store, Mary gave a drink to"

clean_tokens = model.to_tokens(CLEAN_PROMPT)
corrupted_tokens = model.to_tokens(CORRUPTED_PROMPT)

mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")

with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_ld = (clean_logits[0, -1, mary_token] - clean_logits[0, -1, john_token]).item()
corrupted_ld = (corrupted_logits[0, -1, mary_token] - corrupted_logits[0, -1, john_token]).item()

print(f"Clean logit diff:     {clean_ld:+.3f}  (predicts Mary)")
print(f"Corrupted logit diff: {corrupted_ld:+.3f}  (predicts John)")

def normalised_metric(logits):
    ld = logits[0, -1, mary_token] - logits[0, -1, john_token]
    return (ld - corrupted_ld) / (clean_ld - corrupted_ld)

## Direct Path Patching: S-Inhibition → Name-Mover Heads

The IOI circuit (Wang et al. 2022) identifies:
- **S-inhibition heads**: (7,3), (7,9), (8,6), (8,10) — suppress the subject name token
- **Name-mover heads**: (9,9), (9,6), (10,0) — copy the indirect object to the output

Direct path patching lets us measure whether each S-inhibition head communicates *directly* with each name-mover head via the query pathway.

In [ ]:
ioi_src_heads = [(7, 3), (7, 9), (8, 6), (8, 10)]
name_movers = [(9, 9), (9, 6), (10, 0)]

print(f"{"Source":>8}  {"Destination":>12}  {"Score":>8}")
print("-" * 36)

all_results = {}
for sl, sh in ioi_src_heads:
    with torch.no_grad():
        results = get_act_patch_direct_path(
            model=model,
            corrupted_tokens=corrupted_tokens,
            clean_cache=clean_cache,
            corrupted_cache=corrupted_cache,
            patching_metric=normalised_metric,
            src_layer=sl,
            src_head=sh,
            component="q",
            verbose=False,
        )
    all_results[(sl, sh)] = results
    for dl, dh in name_movers:
        if dl > sl:
            score = results[dl, dh].item()
            print(f"  ({sl},{sh:2d})   ({dl},{dh:2d})   {score:+.4f}")

## Results and Interpretation

The results confirm the IOI circuit structure at the **edge level**:

1. **(8,6) → (9,9)** is the strongest single direct path (+0.083). Head 8.6 is the most influential S-inhibition head.
2. All S-inhibition heads show their strongest direct paths running into the known name-mover heads (9.9, 9.6, 10.0).
3. Standard activation patching would show that layer 9 matters — but cannot distinguish *which* upstream head is responsible for each name-mover head's query input.

Direct path patching adds that resolution, isolating the A → B causal edge without affecting any other component's view of A's output.